# Training YOLOv8 Smart Gate — Kaggle

Versi Kaggle dari `a.ipynb`. Perbedaan dengan versi Colab:

- GPU dipilih dari **Settings > Accelerator > GPU T4** (bukan menu Runtime).
- Tanpa GPU, notebook tetap berjalan sebagai **CPU smoke test** (8 epoch,
  416px) — bukan hasil final; training penuh tetap butuh GPU (Kaggle/Colab).
- Dataset dibaca dari **Kaggle Dataset** (`/kaggle/input`) bila tersedia;
  fallback otomatis unduh dari Google Drive (perlu Internet ON).
- Semua kerja & artefak ditulis ke `/kaggle/working` — ambil hasil lewat
  **Save Version (Commit)** lalu tab **Output**.
- Callback `TARGET_MAP50 = 0.98` (early stop) tetap aktif.

`dataset_merged/` tetap dataset utama; `plat-nomor/` fallback.


In [ ]:
# === Setup Kaggle ===
# 1. Kanan-atas > Settings > Accelerator > GPU T4 (atau GPU T4 x2), lalu Save & Run All.
# 2. Settings > Internet > ON untuk pip (bila muncul error "name resolution",
#    Internet masih OFF). Arsip dataset dua cara:
#    A. Kaggle Dataset (Add Data > Datasets) -> data tidak butuh internet, atau
#    B. Diunduh otomatis dari Google Drive (butuh Internet ON).
import importlib.util, subprocess, sys

NEEDED = {
    'ultralytics': 'ultralytics',
    'easyocr': 'easyocr',
    'cv2': 'opencv-python-headless',
    'yaml': 'pyyaml',
    'onnx': 'onnx',
    'onnxruntime': 'onnxruntime',
    'gdown': 'gdown',
}
missing = [pkg for mod, pkg in NEEDED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print('Memasang paket yang belum ada:', missing)
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            'Pemasangan paket gagal - kernel tampaknya tanpa internet. '
            'Nyalakan Settings (panel kanan) > Internet > ON, lalu jalankan ulang sel ini. '
            'Tanpa internet, seluruh paket di atas harus sudah tersedia di image kernel.'
        ) from exc
else:
    print('Semua paket sudah tersedia di image kernel - tidak perlu internet untuk pip.')

import torch
if not torch.cuda.is_available():
    print('PERINGATAN: GPU tidak aktif — kemungkinan penyebab:')
    print('  1. Accelerator belum dipilih: Settings > Accelerator > GPU T4, lalu Save & Run All.')
    print('  2. Verifikasi nomor HP belum dilakukan: kaggle.com/settings > Phone Verification.')
    print('  3. Kuota jam GPU mingguan habis (lihat sisa kuota di panel Settings).')
    print('Notebook tetap berjalan: training otomatis ke mode CPU smoke test (8 epoch, 416px).')
    print('Untuk training penuh tanpa GPU, jalankan a.ipynb di Google Colab (T4 gratis).')
else:
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# --- Cek dataset gabungan jadi (hasil tools/merge_datasets.py, root plat-nomor) ---
from pathlib import Path as _Path
INPUT_ROOT = _Path('/kaggle/input')

def _find_merged():
    for yaml_path in INPUT_ROOT.rglob('data.yaml'):
        root = yaml_path.parent
        if (root / 'train').is_dir() and (root / 'valid').is_dir() and (root / 'test').is_dir():
            return root
    return None

_merged = _find_merged()
if _merged is not None:
    PREMERGED = True
    DATA_ROOT = _merged
    DATA_YAML = _merged / 'data.yaml'
    RUNTIME_YAML = DATA_YAML  # data.yaml sudah memakai path lokal
    print('Dataset gabungan terdeteksi:', DATA_ROOT)
else:
    PREMERGED = False
    from pathlib import Path
    import shutil, zipfile

    # /kaggle/input hanya-baca -> ekstrak ke /kaggle/working supaya merge & training bisa menulis.
    INPUT_ROOT = Path('/kaggle/input')
    SOURCE_ROOT = Path('/kaggle/working/smartgate-sources')
    SOURCE_ROOT.mkdir(parents=True, exist_ok=True)

    # Arsip training (ID Google Drive dipakai hanya bila dataset tidak ada di /kaggle/input).
    DRIVE_FILES = {
        'smartgate-dataset-merged.zip': '1mZxUtYrBLjsXVrzKvoZDhia9PZ9Y14AG',
    }

    def find_input(name):
        matches = list(INPUT_ROOT.rglob(name))
        return matches[0] if matches else None

    zip_files = []
    for filename, file_id in DRIVE_FILES.items():
        local = find_input(filename)  # opsi A: Kaggle Dataset
        if local is None:             # opsi B: unduh dari Drive (butuh internet)
            local = SOURCE_ROOT / filename
            if not local.is_file() or local.stat().st_size < 1024:
                try:
                    import gdown
                    gdown.download(id=file_id, output=str(local), quiet=False)
                except Exception as exc:
                    raise RuntimeError(
                        f'{filename} tidak ada di /kaggle/input dan unduhan gagal. '
                        'Unggah arsip sebagai Kaggle Dataset (Add Data > Datasets) '
                        'atau nyalakan Internet (Settings > Internet > ON).'
                    ) from exc
        if not Path(local).is_file() or Path(local).stat().st_size < 1024:
            raise RuntimeError(f'Tidak tersedia / terlalu kecil: {filename}')
        zip_files.append(Path(local))

    # Ekstrak tiap ZIP ke folder terpisah agar sumber dan anotasi tidak tertimpa.
    for zip_path in zip_files:
        target = SOURCE_ROOT / zip_path.stem
        marker = target / '.extracted-ok'
        if not marker.exists():
            if target.exists():
                shutil.rmtree(target)
            target.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(zip_path) as archive:
                bad_member = archive.testzip()
                if bad_member is not None:
                    raise RuntimeError(f'ZIP rusak pada {zip_path.name}, entry: {bad_member}')
                archive.extractall(target)
            marker.write_text('ok', encoding='utf-8')

    main_root = SOURCE_ROOT / 'smartgate-dataset-merged'
    candidates = [p.parent for p in main_root.rglob('data.yaml') if (p.parent / 'train').is_dir()]
    preferred = [p for p in candidates if p.name == 'plat-nomor']
    DATA_ROOT = next(iter(preferred or candidates), None)
    if DATA_ROOT is None:
        raise FileNotFoundError('data.yaml tidak ditemukan setelah ekstraksi smartgate-dataset-merged.zip.')
    DATA_YAML = DATA_ROOT / 'data.yaml'
    print('Arsip sumber:')
    for item in zip_files:
        print(f'  - {item.name}: {item.stat().st_size / 1024**2:.2f} MB')
    print('Dataset utama:', DATA_ROOT)
    print('Config sumber:', DATA_YAML)


In [ ]:
if PREMERGED:
    print('Dataset sudah berupa gabungan final - tahap merge dilewati.')
else:
    import yaml, hashlib, shutil, json

    config = yaml.safe_load(DATA_YAML.read_text())
    config['path'] = str(DATA_ROOT)
    config['train'] = 'train/images'
    config['val'] = 'valid/images'
    config['test'] = 'test/images'
    config['nc'] = 1
    config['names'] = ['number-plate']
    RUNTIME_YAML = Path('/kaggle/working/dataset_runtime.yaml')
    RUNTIME_YAML.write_text(yaml.safe_dump(config, sort_keys=False))
    print(RUNTIME_YAML.read_text())

    IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    train_img_dir = DATA_ROOT / 'train' / 'images'
    train_lbl_dir = DATA_ROOT / 'train' / 'labels'
    val_test_stems = {p.stem for split in ('valid', 'test') for p in (DATA_ROOT / split / 'images').iterdir() if p.is_file()}

    def file_hash(path):
        h = hashlib.sha256()
        with open(path, 'rb') as f:
            for chunk in iter(lambda: f.read(1 << 20), b''):
                h.update(chunk)
        return h.hexdigest()

    val_test_hashes = {file_hash(p) for split in ('valid', 'test') for p in (DATA_ROOT / split / 'images').iterdir() if p.is_file()}

    summary = {'roboflow': 0, 'kaggle': 0, 'background': 0, 'skipped_leakage': 0, 'invalid': 0}

    def valid_yolo_lines(raw_lines):
        out = []
        for line in raw_lines:
            line = line.strip()
            if not line:
                continue
            f = line.split()
            if len(f) != 5:
                return None
            try:
                cls = int(f[0]); vals = [float(x) for x in f[1:]]
            except ValueError:
                return None
            if cls != 0 or any(v < 0 or v > 1 for v in vals):
                return None
            out.append(line)
        return out

    def add_image(img_path, lines, prefix, counter):
        if img_path.stem in val_test_stems or file_hash(img_path) in val_test_hashes:
            summary['skipped_leakage'] += 1
            return
        stem = f'{prefix}_{counter:05d}'
        shutil.copy2(img_path, train_img_dir / f'{stem}{img_path.suffix.lower()}')
        (train_lbl_dir / f'{stem}.txt').write_text('\n'.join(lines) + ('\n' if lines else ''))

    # --- Roboflow: hanya split train (valid/test utama tidak disentuh) ---
    rf_root = SOURCE_ROOT / 'Indonesia License Plate Detection.v1i.yolov8'
    if (rf_root / 'train' / 'images').is_dir():
        i = 0
        for img in sorted((rf_root / 'train' / 'images').iterdir()):
            if img.suffix.lower() not in IMG_EXTS:
                continue
            lbl = rf_root / 'train' / 'labels' / f'{img.stem}.txt'
            if not lbl.is_file():
                summary['invalid'] += 1
                continue
            lines = valid_yolo_lines(lbl.read_text().splitlines())
            if lines is None:
                summary['invalid'] += 1
                continue
            add_image(img, lines, 'rf', i); i += 1
            summary['roboflow'] += 1

    # --- Kaggle: konversi COCO -> YOLO ---
    kg_root = SOURCE_ROOT / 'Kaggle plate dataset'
    coco_files = sorted(kg_root.rglob('*.json'))
    coco_path = next((p for p in coco_files if any(k in p.name.lower() for k in ('instances', 'coco', 'annotation'))), coco_files[0] if coco_files else None)
    if coco_path is not None:
        coco = json.loads(coco_path.read_text())
        images_meta = {im['id']: im for im in coco.get('images', [])}
        by_img = {}
        for a in coco.get('annotations', []):
            by_img.setdefault(a['image_id'], []).append(a)
        plate_cats = [c['id'] for c in coco.get('categories', []) if 'plate' in c['name'].lower()] or [c['id'] for c in coco.get('categories', [])]
        files = {Path(im['file_name']).name for im in images_meta.values()}
        img_dirs = [p for p in kg_root.rglob('images') if p.is_dir()]
        img_dir = next((d for d in img_dirs if any((d / fn).exists() for fn in list(files)[:20])), img_dirs[0] if img_dirs else None)
        if img_dir is not None:
            i = 0
            for mid, im in sorted(images_meta.items()):
                img_path = img_dir / Path(im['file_name']).name
                if not img_path.exists():
                    summary['invalid'] += 1
                    continue
                W, H = float(im.get('width', 0) or 0), float(im.get('height', 0) or 0)
                lines, ok = [], True
                for a in by_img.get(mid, []):
                    if a.get('category_id') not in plate_cats:
                        continue
                    x, y, w, h = (float(v) for v in a['bbox'])
                    if W and H:
                        cx, cy, bw, bh = (x + w / 2) / W, (y + h / 2) / H, w / W, h / H
                    else:
                        ok = False; break
                    if not (0 <= cx <= 1 and 0 <= cy <= 1 and 0 < bw <= 1 and 0 < bh <= 1):
                        ok = False; break
                    lines.append(f'0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')
                if not ok or not lines:
                    summary['invalid'] += 1
                    continue
                add_image(img_path, lines, 'kg', i); i += 1
                summary['kaggle'] += 1

    # --- Background no-plate: label kosong ke train saja ---
    bg_root = SOURCE_ROOT / 'no-plate-backgrounds'
    if (bg_root / 'images').is_dir():
        i = 0
        for img in sorted((bg_root / 'images').iterdir()):
            if img.suffix.lower() not in IMG_EXTS:
                continue
            add_image(img, [], 'bg', i); i += 1
            summary['background'] += 1

    print('MERGE SUMMARY:', summary)

    # --- OCR reference data (crop plat + label teks) ---
    OCR_TRAIN_ROOT = Path('/kaggle/working/ocr_reference')
    plate_text = kg_root / 'plate_text_dataset'
    if plate_text.is_dir() and not OCR_TRAIN_ROOT.exists():
        shutil.copytree(plate_text, OCR_TRAIN_ROOT)
        print('OCR reference:', OCR_TRAIN_ROOT)

    # --- Validasi seluruh split (label kosong = background, diizinkan) ---
    for split in ('train', 'val', 'test'):
        image_dir = DATA_ROOT / config[split]
        label_dir = DATA_ROOT / config[split].replace('/images', '/labels')
        if not image_dir.is_dir() or not label_dir.is_dir():
            raise FileNotFoundError(f'{split}: struktur tidak ditemukan di {image_dir}')
        images = {p.stem for p in image_dir.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS}
        labels = {p.stem for p in label_dir.glob('*.txt')}
        missing = sorted(images - labels)
        orphan = sorted(labels - images)
        if missing or orphan:
            raise ValueError(f'{split}: missing labels={len(missing)}, orphan labels={len(orphan)}')
        bg = box = 0
        for lp in label_dir.glob('*.txt'):
            nonempty = [l.strip() for l in lp.read_text().splitlines() if l.strip()]
            if not nonempty:
                bg += 1
                continue
            for ln, line in enumerate(nonempty, 1):
                f = line.split()
                if len(f) != 5 or int(f[0]) != 0 or not all(0.0 <= float(v) <= 1.0 for v in f[1:]):
                    raise ValueError(f'Anotasi invalid: {lp}:{ln} -> {line}')
                box += 1
        print(f'{split}: {len(images)} images ({bg} background, {box} kotak), valid')
    print('Struktur dan anotasi dataset valid.')


In [ ]:
def count_labels(folder):
    files = list(Path(folder).glob('*.txt'))
    objects = sum(len(path.read_text().splitlines()) for path in files)
    return len(files), objects

for split in ('train', 'valid', 'test'):
    label_dir = DATA_ROOT / split / 'labels'
    files, objects = count_labels(label_dir)
    print(f'{split}: {files} file label, {objects} anotasi')

## Training

Pastikan GPU aktif (Settings > Accelerator > GPU T4). Kotak anotasi harus tepat
mengelilingi papan plat, bukan bumper, kamera, gril, atau seluruh kendaraan.
Training berhenti otomatis bila mAP50 menyentuh `TARGET_MAP50` (0,98) atau
`patience=15` epoch tanpa perbaikan.

**Perbandingan YOLOv8n vs YOLOv11n:** jalankan notebook dua kali dengan
`BASE_MODEL = 'yolov8n.pt'` lalu `'yolo11n.pt'` (nama run otomatis
`smartgate-kaggle-v6-yolov8n` / `-yolo11n`), bandingkan kedua
`evaluation_summary.json`. Catatan literatur (Ramadhani dkk., 2026, Jurnal
Teknika 18(2), DOI 10.30736/jt.v18i2.1604): mAP deteksi tinggi tidak
otomatis berarti akurasi OCR lebih baik — nilai juga valid-format rate dan
latensi.


In [ ]:
from ultralytics import YOLO
import torch

device = 0 if torch.cuda.is_available() else 'cpu'
batch = 16 if torch.cuda.is_available() else 8
epochs = 50 if torch.cuda.is_available() else 8
imgsz = 640 if torch.cuda.is_available() else 416
BASE_MODEL = 'yolov8n.pt'  # ganti 'yolo11n.pt' untuk perbandingan v8n vs yolo11n
MODEL_TAG = BASE_MODEL.replace('.pt', '')
if device == 'cpu':
    print('Mode CPU smoke test (8 epoch, 416px) — bukan hasil final.')
    print('Training penuh butuh GPU: verifikasi HP / cek kuota Kaggle, atau pakai Colab T4.')
BASE_MODEL = 'yolov8n.pt'
MODEL_TAG = BASE_MODEL.replace('.pt', '')
EXPERIMENT_NAME = f'smartgate-colab-v6-{MODEL_TAG}'
print('Device:', device)
print('Eksperimen:', EXPERIMENT_NAME)

RUN_DIR = DATA_ROOT.parent / 'runs' / 'detect'
model = YOLO(BASE_MODEL)

# Early stop kustom: berhenti begitu target mAP50 tercapai (hemat waktu GPU/Colab).
TARGET_MAP50 = 0.98

def stop_at_target(trainer):
    map50 = (trainer.metrics or {}).get('metrics/mAP50(B)', 0.0)
    if map50 >= TARGET_MAP50:
        trainer.stop = True
        print(f'\nTarget mAP50 {TARGET_MAP50:.2f} tercapai ({map50:.4f}) - training dihentikan.')

model.add_callback('on_fit_epoch_end', stop_at_target)
model.train(
    data=str(RUNTIME_YAML),
    epochs=epochs,
    imgsz=imgsz,
    batch=batch,
    device=device,
    project=str(RUN_DIR),
    name=EXPERIMENT_NAME,
    patience=15,
    cos_lr=True,
    box=7.5,
    hsv_h=0.015,
    hsv_s=0.55,
    hsv_v=0.35,
    degrees=5.0,
    translate=0.08,
    scale=0.35,
    shear=0.0,
    perspective=0.0,
    fliplr=0.0,
    mosaic=0.7,
    mixup=0.05,
    close_mosaic=10,
    pretrained=True,
    plots=True,
    val=True,
    seed=42,
    workers=2,
    cache=False,
    exist_ok=True,
)
print('Training selesai.')


In [ ]:
best_pt = RUN_DIR / EXPERIMENT_NAME / 'weights' / 'best.pt'
if not best_pt.is_file():
    raise FileNotFoundError(best_pt)
trained = YOLO(str(best_pt))
val_metrics = trained.val(data=str(RUNTIME_YAML), split='val', imgsz=imgsz, device=device)
test_metrics = trained.val(data=str(RUNTIME_YAML), split='test', imgsz=imgsz, device=device)
if float(test_metrics.box.map50) < 0.98:
    print('Menjalankan evaluasi test dengan TTA...')
    test_metrics_tta = trained.val(data=str(RUNTIME_YAML), split='test', imgsz=imgsz, device=device, augment=True)
    if float(test_metrics_tta.box.map50) > float(test_metrics.box.map50):
        test_metrics = test_metrics_tta
print('best.pt:', best_pt)
print(f'Validation mAP50: {val_metrics.box.map50:.4f}')
print(f'Validation mAP50-95: {val_metrics.box.map:.4f}')
print(f'Test mAP50: {test_metrics.box.map50:.4f}')
print(f'Test mAP50-95: {test_metrics.box.map:.4f}')

In [ ]:

import cv2
import re
import time
import json
import numpy as np
import easyocr

ocr_reader = easyocr.Reader(['en'], gpu=torch.cuda.is_available())
ALLOWLIST = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'
DIGIT_TO_CHAR = {'0': 'O', '1': 'I', '2': 'Z', '3': 'B', '4': 'A', '5': 'S', '6': 'G', '7': 'T', '8': 'B', '9': 'Q'}
CHAR_TO_DIGIT = {'O': '0', 'D': '0', 'Q': '0', 'I': '1', 'L': '1', 'Z': '2', 'A': '4', 'S': '5', 'G': '6', 'T': '7', 'B': '8'}
# V4: kode wilayah Indonesia tidak memakai O/Q/I/L/angka. Makassar DD sering terbaca 00/O0 -> D
PREFIX_FIX = {'0': 'D', 'O': 'D', 'Q': 'D', 'I': 'D', 'L': 'D', '8': 'B'}
VALID_PREFIXES = {'A','B','D','E','F','G','H','K','L','M','N','P','R','S','T','W','Z','AA','AB','AD','AE','AG','BA','BB','BD','BE','BG','BH','BK','BL','BM','BN','BP','DA','DB','DC','DD','DE','DG','DH','DK','DL','DM','DN','DP','DR','DT','DW','EA','EB','ED','KB','KH','KT','KU','PA','PB'}

def assess_crop_quality(crop_bgr):
    gray = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2GRAY)
    blur_score = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    brightness = float(np.mean(gray))
    return {'blur_score': round(blur_score, 2), 'brightness': round(brightness, 2), 'is_usable': blur_score >= 30.0 and 20.0 <= brightness <= 245.0}

def detect_plate_color(crop_bgr):
    hsv = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2HSV)
    h, w = hsv.shape[:2]
    center = hsv[int(h * .20):int(h * .75), int(w * .15):int(w * .85)]
    total = center.shape[0] * center.shape[1] + 1e-5
    yellow = cv2.inRange(center, (15, 80, 80), (35, 255, 255))
    red = cv2.bitwise_or(cv2.inRange(center, (0, 80, 80), (10, 255, 255)), cv2.inRange(center, (160, 80, 80), (180, 255, 255)))
    if np.sum(yellow > 0) / total > .25:
        return 'YELLOW'
    if np.sum(red > 0) / total > .25:
        return 'RED'
    return 'WHITE' if float(np.mean(center[:, :, 2])) > 127 else 'BLACK'

def preprocess_plate_for_ocr(crop_bgr, plate_color):
    h, w = crop_bgr.shape[:2]
    x_trim = int(w * .03)
    y_top = int(h * .04)
    y_bot = max(y_top + 10, int(h * .70))
    zone = crop_bgr[y_top:y_bot, x_trim:max(x_trim + 10, w - x_trim)]
    mh, mw = zone.shape[:2]
    if mw < 260:
        zone = cv2.resize(zone, None, fx=260.0 / max(mw, 1), fy=260.0 / max(mw, 1), interpolation=cv2.INTER_CUBIC)
    gray = cv2.cvtColor(zone, cv2.COLOR_BGR2GRAY)
    if plate_color == 'BLACK' or float(np.mean(gray)) < 110:
        gray = cv2.bitwise_not(gray)
    # V4: CLAHE lembut agar garis moire/pantulan dan baut tidak menonjol jadi karakter
    enhanced = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(8, 8)).apply(gray)
    return cv2.copyMakeBorder(enhanced, 6, 6, 8, 8, cv2.BORDER_REPLICATE)

def extract_main_ocr_text(ocr_out, img_h):
    if not ocr_out:
        return '', 0.0
    info = []
    for bbox, text, conf in ocr_out:
        ys = [pt[1] for pt in bbox]
        xs = [pt[0] for pt in bbox]
        clean = re.sub(r'[^A-Z0-9]', '', text.upper())
        if clean:
            info.append({'x_min': min(xs), 'y_center': float(np.mean(ys)), 'height': max(ys) - min(ys), 'text': clean, 'conf': float(conf)})
    if not info:
        return '', 0.0
    # V4: dominan-line filter - buang baris teks pajak di bawah plat (y >= 0.78h)
    main = [it for it in info if it['y_center'] <= .78 * img_h] or info
    max_h = max(it['height'] for it in main)
    main = [it for it in main if it['height'] >= .45 * max_h] or main
    main.sort(key=lambda it: it['x_min'])
    return ''.join(it['text'] for it in main), float(np.mean([it['conf'] for it in main]))

def format_plate(raw_text):
    clean = re.sub(r'[^A-Z0-9]', '', raw_text.upper())
    if len(clean) < 3:
        return clean, False
    # V4: buang awalan I/1/L/0/O yang terdeteksi sebelum kode wilayah
    if len(clean) >= 6 and clean[0] in ('I', '1', 'L', '0', 'O'):
        candidate = ''.join(PREFIX_FIX.get(c, c) for c in clean[1:3])
        if candidate in VALID_PREFIXES:
            clean = clean[1:]
    best = None; best_score = -999
    for start in range(min(2, len(clean))):
        for length in range(min(9, len(clean) - start), 2, -1):
            sub = clean[start:start + length]
            for p_len in (2, 1):
                if len(sub) <= p_len:
                    continue
                for d_len in range(min(4, len(sub) - p_len), 0, -1):
                    s_len = len(sub) - p_len - d_len
                    if s_len < 0 or s_len > 3:
                        continue
                    p_raw = sub[:p_len]; d_raw = sub[p_len:p_len + d_len]; s_raw = sub[p_len + d_len:]
                    p = ''.join(PREFIX_FIX.get(c, c) for c in p_raw)
                    d = ''.join(CHAR_TO_DIGIT.get(c, c) for c in d_raw)
                    s = ''.join(DIGIT_TO_CHAR.get(c, c) for c in s_raw)
                    if not (p.isalpha() and d.isdigit() and (not s or s.isalpha())) or d[0] == '0' or p not in VALID_PREFIXES:
                        continue
                    score = (sum(4 for a, b in zip(p_raw, p) if a == b) + sum(5 for a, b in zip(d_raw, d) if a == b) + sum(4 for a, b in zip(s_raw, s) if a == b) - start * 8)
                    if 2 <= d_len <= 4 and 1 <= s_len <= 3:
                        score += 12
                    if score > best_score:
                        best_score = score; best = f'{p} {d} {s}'.strip()
    return (best, True) if best is not None and best_score >= 30 else (clean, False)

def plate_passes(crop_bgr):
    # V4 multi-pass: crop penuh + crop dengan pinggir lebih banyak dibuang (kisi bingkai)
    h, w = crop_bgr.shape[:2]
    passes = [crop_bgr]
    x = int(w * .08)
    if w - 2 * x > 40:
        passes.append(crop_bgr[:, x:w - x])
    return passes

def read_smartgate_plate(image_bgr, yolo_model, conf_thres=.35):
    started = time.perf_counter()
    result = yolo_model.predict(image_bgr, conf=conf_thres, imgsz=imgsz, verbose=False)[0]
    detections = []
    h_img, w_img = image_bgr.shape[:2]
    for box in result.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        det_conf = float(box.conf[0])
        crop = image_bgr[max(0, y1):min(h_img, y2), max(0, x1):min(w_img, x2)]
        if crop.size == 0:
            continue
        quality = assess_crop_quality(crop)
        reads = []
        for variant in plate_passes(crop):
            if variant.size == 0:
                continue
            color = detect_plate_color(variant)
            processed = preprocess_plate_for_ocr(variant, color)
            ocr_out = ocr_reader.readtext(processed, allowlist=ALLOWLIST, detail=1, paragraph=False)
            raw, ocr_conf = extract_main_ocr_text(ocr_out, processed.shape[0])
            plate, valid = format_plate(raw)
            reads.append({'plate': plate, 'valid': valid, 'conf': ocr_conf, 'raw': raw, 'color': color})
        if not reads:
            continue
        valids = [r for r in reads if r['valid']]
        best = max(valids, key=lambda r: r['conf']) if valids else max(reads, key=lambda r: r['conf'])
        detections.append({'bbox': [x1, y1, x2, y2], 'plate_number': best['plate'], 'raw_ocr': best['raw'], 'is_valid_format': best['valid'], 'plate_color': best['color'], 'det_conf': round(det_conf, 4), 'ocr_conf': round(best['conf'], 4), 'quality': quality})
    return {'latency_ms': round((time.perf_counter() - started) * 1000, 2), 'plates': detections}

test_image_dir = DATA_ROOT / 'test' / 'images'
test_images = sorted(p for p in test_image_dir.iterdir() if p.is_file())
latencies = []; valid_reads = 0; detected = 0; sample_outputs = []
for img_path in test_images:
    image = cv2.imread(str(img_path))
    if image is None:
        continue
    result = read_smartgate_plate(image, trained)
    latencies.append(result['latency_ms'])
    detected += len(result['plates'])
    valid_reads += sum(1 for item in result['plates'] if item['is_valid_format'])
    if len(sample_outputs) < 10 and result['plates']:
        sample_outputs.append({'file': img_path.name, **result['plates'][0]})
onnx_path = trained.export(format='onnx', imgsz=imgsz, simplify=True, dynamic=False)
test_map50 = float(test_metrics.box.map50)
report = {'model': str(best_pt), 'onnx_model': str(onnx_path), 'validation': {'mAP50': float(val_metrics.box.map50), 'mAP50_95': float(val_metrics.box.map)}, 'test': {'images': len(test_images), 'mAP50': test_map50, 'mAP50_95': float(test_metrics.box.map)}, 'ocr_and_gate_benchmark': {'total_detected_plates': detected, 'valid_indonesian_format_count': valid_reads, 'valid_format_rate': round(valid_reads / max(detected, 1), 4), 'avg_end_to_end_latency_ms': round(float(np.mean(latencies)), 2) if latencies else 0.0, 'p95_end_to_end_latency_ms': round(float(np.percentile(latencies, 95)), 2) if latencies else 0.0}, 'detector_target_mAP50': .98, 'detector_target_met': test_map50 >= .98, 'production_kpi_status': 'NOT_CLAIMED', 'sample_reads': sample_outputs}
print('=== LAPORAN EVALUASI DETECTOR & OCR SMART GATE (V4 PARSER) ===')
print(f'Validation mAP50    : {report["validation"]["mAP50"]:.4f}')
print(f'Validation mAP50-95 : {report["validation"]["mAP50_95"]:.4f}')
print(f'Test mAP50          : {report["test"]["mAP50"]:.4f}')
print(f'Test mAP50-95       : {report["test"]["mAP50_95"]:.4f}')
print(f'Valid Format Rate   : {report["ocr_and_gate_benchmark"]["valid_format_rate"]:.2%} ({valid_reads}/{detected})')
print(f'Avg latency         : {report["ocr_and_gate_benchmark"]["avg_end_to_end_latency_ms"]} ms')
print(f'P95 latency         : {report["ocr_and_gate_benchmark"]["p95_end_to_end_latency_ms"]} ms')
print(f'Detector target 98% : {"TERCAPAI" if report["detector_target_met"] else "BELUM TERCAPAI"}')
summary_path = RUN_DIR / EXPERIMENT_NAME / 'evaluation_summary.json'
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary_path.write_text(json.dumps(report, indent=2))
print('Ringkasan tersimpan:', summary_path)


In [ ]:
import shutil

ARTIFACT_DIR = RUN_DIR / EXPERIMENT_NAME
REPORT_PATH = ARTIFACT_DIR / 'evaluation_summary.json'
required_files = [best_pt, REPORT_PATH]
missing = [str(path) for path in required_files if not path.is_file()]
if missing:
    raise FileNotFoundError('Artefak belum tersedia: ' + ', '.join(missing))

DOWNLOAD_DIR = Path('/kaggle/working/smartgate-artifacts')
DOWNLOAD_DIR.mkdir(exist_ok=True)
shutil.copy2(best_pt, DOWNLOAD_DIR / 'best.pt')
shutil.copy2(REPORT_PATH, DOWNLOAD_DIR / 'evaluation_summary.json')
for result_name in ('results.csv', 'confusion_matrix.png', 'PR_curve.png', 'F1_curve.png', 'best.onnx'):
    result_path = (Path(onnx_path) if result_name == 'best.onnx' else ARTIFACT_DIR / result_name)
    if result_path.is_file():
        shutil.copy2(result_path, DOWNLOAD_DIR / result_name)

ARCHIVE_PATH = Path('/kaggle/working/smartgate-artifacts.zip')
if ARCHIVE_PATH.exists():
    ARCHIVE_PATH.unlink()
shutil.make_archive(str(ARCHIVE_PATH.with_suffix('')), 'zip', DOWNLOAD_DIR)
if not ARCHIVE_PATH.is_file() or ARCHIVE_PATH.stat().st_size == 0:
    raise RuntimeError(f'ZIP artefak gagal dibuat: {ARCHIVE_PATH}')
print('Artefak siap:', ARCHIVE_PATH)
print(f'Ukuran ZIP: {ARCHIVE_PATH.stat().st_size / 1024**2:.2f} MB')
print('Mengambil hasil: klik "Save Version" (Commit) di kanan atas — setelah commit')
print('selesai, buka tab Output/Output notebook untuk mengunduh smartgate-artifacts.zip.')


## Setelah Training

1. Jalankan seluruh sel sampai artefak (`smartgate-artifacts.zip`) dibuat.
2. Klik **Save Version (Commit)** — setelah selesai, unduh ZIP dari tab **Output**.
3. Salin `best.pt` ke `Tugas Citra Digital/prototype/runs/detect/smartgate-kaggle-v6/weights/best.pt`.
4. Jalankan aplikasi dengan `SGS_YOLO_WEIGHTS` menunjuk ke file tersebut, lalu restart Flask.
